# Chronological median baseline

Predict delivery duration in minutes for orders placed after the training period. This notebook runs the same code as `python -m ml.scripts.train`. Use the backend Python environment as the kernel and install its requirements plus `ipykernel`.

The prepared Parquet comes from the [data audit and ingestion workflow](01_data_audit.ipynb). The [dataset configuration](../configs/dataset.yaml) defines the target and order-time feature allowlist.

In [1]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
backend = next(
    (
        candidate
        for parent in (cwd, *cwd.parents)
        for candidate in (parent, parent / "apps" / "backend")
        if (candidate / "ml" / "configs" / "dataset.yaml").is_file()
        and (candidate / "app" / "main.py").is_file()
    ),
    None,
)
if backend is None:
    raise FileNotFoundError("Open this notebook from within the project repository.")
if str(backend) not in sys.path:
    sys.path.insert(0, str(backend))

from ml.manifest import resolve_dataset_paths
from ml.train import run_training

paths = resolve_dataset_paths()
print(f"Dataset: {paths.processed_parquet.relative_to(backend).as_posix()}")
result = run_training()

Dataset: ml/data/processed/dataset.parquet


## Define the chronological split

The workflow checks `order_timestamp`, counts unusable timestamps, and sorts the remaining orders from oldest to newest. It chooses cut points near 70% and 85% of the usable rows. Each cut falls between timestamp groups and leaves all three partitions populated, so orders with the same timestamp stay together. For equally close cut points the earlier one wins.

The oldest partition fits the model, the middle partition measures performance during model selection, and the newest partition is held for final evaluation. The ranges below are inclusive and use the dataset's local clock time. Percentages use the rows with usable timestamps.

In [2]:
from IPython.display import display

split = result.split
print(
    f"Excluded timestamps: {split.missing_timestamps:,} missing, "
    f"{split.invalid_timestamps:,} invalid."
)
display(split.summary())

Excluded timestamps: 1,731 missing, 0 invalid.


,Rows,Share (%),First order,Last order
Training,30698,70.00,2022-02-11 08:10:00,2022-03-25 17:20:00
Validation,6580,15.00,2022-03-25 17:25:00,2022-03-31 20:50:00
Test,6575,14.99,2022-03-31 20:55:00,2022-04-07 00:00:00


## Define model inputs and the target

`X` contains the `order-time` fields on the feature allowlist. `y` contains `delivery_time_minutes`. Order timestamps determine the split. Identifiers, pickup fields, and the target are excluded from `X`.

In [ ]:
features = list(result.feature_columns)
target = result.target_column

X_train = split.train.loc[:, features]
y_train = split.train[target]
X_validation = split.validation.loc[:, features]
y_validation = split.validation[target]

print(f"X columns: {', '.join(X_train.columns)}")
print(f"y column: {y_train.name}")
print(f"Training: X {X_train.shape}, y {y_train.shape}")
print(f"Validation: X {X_validation.shape}, y {y_validation.shape}")

X columns: Type_of_order, Festival, City, order_hour, order_day_of_week, distance_km
y column: delivery_time_minutes
Training: X (30698, 6), y (30698,)
Validation: X (6580, 6), y (6580,)


## Read the baseline MAE

The median dummy regressor gives every order the same estimate, the median delivery time among training orders. It ignores the input feature values. This gives future models a reference score without fitting any imputation, encoding, or scaling. [scikit-learn DummyRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyRegressor.html)

Validation mean absolute error (MAE) averages the absolute difference between predicted and actual delivery time. An MAE of about 7.75 means the baseline missed by about 7.75 minutes per validation order on average.

In [4]:
print(f"Training median: {result.training_median:.6f} minutes")
print(f"Validation MAE: {result.validation_mae:.6f} minutes")
print("Test set reserved for final evaluation.")

Training median: 26.000000 minutes
Validation MAE: 7.752280 minutes
Test set reserved for final evaluation.


## Compare future models

Compare Linear Regression, a Decision Tree, and a Random Forest with the median baseline on the same chronological validation partition. Fit each model's preprocessing on the training rows. Choose the model with the lowest validation MAE and evaluate the winner once on the held-out test set.

A Random Forest is a candidate because trees can learn nonlinear effects and interactions. For example, with confirmed order-time traffic and weather inputs, a tree could learn that road congestion affects long routes differently during a storm. Averaging the trees' predictions reduces sensitivity to any one tree. The current allowlist includes distance, and traffic and weather remain excluded until their order-time availability is confirmed. [scikit-learn RandomForestRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html)

With the default squared-error criterion, the forest's trees favor splits that reduce squared prediction errors within the resulting groups. Validation MAE measures the average miss in minutes on later orders and is the criterion for comparing models here. Root mean squared error (RMSE) also has units of minutes and gives large misses more weight. If RMSE is substantially higher than MAE, inspect the largest residuals to see whether a few badly estimated deliveries are driving the error. [scikit-learn regression metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics)